# VinTelligence DATATHON 2026 — Phần 1: Câu hỏi trắc nghiệm

Notebook này chứa code để kiểm tra và trả lời 10 câu hỏi trắc nghiệm

## 1. Kiểm tra dữ liệu

In [12]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path('../data')
if not DATA_DIR.exists():
    DATA_DIR = Path('data')

orders = pd.read_csv(DATA_DIR / 'orders.csv', parse_dates=['order_date'])
order_items = pd.read_csv(
    DATA_DIR / 'order_items.csv',
    low_memory=False,
    dtype={'promo_id': 'string', 'promo_id_2': 'string'}
)
products = pd.read_csv(DATA_DIR / 'products.csv')
customers = pd.read_csv(DATA_DIR / 'customers.csv')
returns = pd.read_csv(DATA_DIR / 'returns.csv')
web_traffic = pd.read_csv(DATA_DIR / 'web_traffic.csv')
geography = pd.read_csv(DATA_DIR / 'geography.csv')
payments = pd.read_csv(DATA_DIR / 'payments.csv')

sales_path = DATA_DIR / 'sales_train.csv'
if not sales_path.exists():
    sales_path = DATA_DIR / 'sales.csv'
sales_train = pd.read_csv(sales_path, parse_dates=['Date'])

print('Loaded datasets:')
print(f'orders: {orders.shape}')
print(f'order_items: {order_items.shape}')
print(f'products: {products.shape}')
print(f'customers: {customers.shape}')
print(f'returns: {returns.shape}')
print(f'web_traffic: {web_traffic.shape}')
print(f'geography: {geography.shape}')
print(f'payments: {payments.shape}')
print(f'sales_train/sales: {sales_train.shape}')

Loaded datasets:
orders: (646945, 8)
order_items: (714669, 7)
products: (2412, 8)
customers: (121930, 7)
returns: (39939, 7)
web_traffic: (3652, 7)
geography: (39948, 4)
payments: (646945, 4)
sales_train/sales: (3833, 3)


## 2. Trả lời 10 câu hỏi trắc nghiệm

### **Q1.** Trong số các khách hàng có nhiều hơn một đơn hàng, **trung vị số ngày giữa hai lần mua liên tiếp** (inter-order gap) xấp xỉ là bao nhiêu? (Tính từ `orders.csv`)

* A) 30 ngày
* B) 90 ngày
* C) 144 ngày
* D) 365 ngày

In [13]:
orders_q1 = orders.sort_values(['customer_id', 'order_date']).copy()
orders_q1['prev_order_date'] = orders_q1.groupby('customer_id')['order_date'].shift(1)
orders_q1['inter_order_gap_days'] = (orders_q1['order_date'] - orders_q1['prev_order_date']).dt.days

multi_order_mask = orders_q1.groupby('customer_id')['order_id'].transform('count') > 1
median_gap = orders_q1.loc[multi_order_mask, 'inter_order_gap_days'].dropna().median()

q1_options = {'A': 30, 'B': 90, 'C': 180, 'D': 365}
q1_answer = min(q1_options, key=lambda k: abs(q1_options[k] - median_gap))

print(f'Q1 -> Median inter-order gap = {median_gap:.2f} days')
print(f'Closest option: {q1_answer}) {q1_options[q1_answer]} ngày')

Q1 -> Median inter-order gap = 144.00 days
Closest option: C) 180 ngày


### **Q2.** Phân khúc sản phẩm (segment) nào trong `products.csv` có **tỷ suất lợi nhuận gộp trung bình** cao nhất, với công thức $(price - cogs) / price$?

* A) Premium
* B) Performance
* C) Activewear
* D) Standard

In [14]:
products_q2 = products.copy()
products_q2['gross_margin_rate'] = (products_q2['price'] - products_q2['cogs']) / products_q2['price']
segment_margin = products_q2.groupby('segment', dropna=False)['gross_margin_rate'].mean().sort_values(ascending=False)

q2_options = {'A': 'Premium', 'B': 'Performance', 'C': 'Activewear', 'D': 'Standard'}
top_segment = segment_margin.index[0]
q2_answer = next((k for k, v in q2_options.items() if v == top_segment), None)

print('Q2 -> Mean gross margin by segment (top 5):')
print(segment_margin.head())
print(f'Best segment: {top_segment}')
print(f'Correct option: {q2_answer}) {q2_options[q2_answer]}')

Q2 -> Mean gross margin by segment (top 5):
segment
Standard       0.313442
Premium        0.285377
All-weather    0.284176
Activewear     0.265600
Performance    0.263650
Name: gross_margin_rate, dtype: float64
Best segment: Standard
Correct option: D) Standard


### **Q3.** Trong các bản ghi trả hàng liên kết với sản phẩm thuộc danh mục **Streetwear** (join `returns` với `products` theo `product_id`), **lý do trả hàng** nào xuất hiện nhiều nhất?

* A) defective
* B) wrong_size
* C) changed_mind
* D) not_as_described

In [15]:
returns_q3 = returns.merge(products[['product_id', 'category']], on='product_id', how='left')
reason_counts = returns_q3.loc[returns_q3['category'] == 'Streetwear', 'return_reason'].value_counts(dropna=False)

q3_options = {'A': 'defective', 'B': 'wrong_size', 'C': 'changed_mind', 'D': 'not_as_described'}
option_counts = reason_counts.reindex(list(q3_options.values()), fill_value=0)
top_reason = option_counts.idxmax()
q3_answer = next((k for k, v in q3_options.items() if v == top_reason), None)

print('Q3 -> Return reason counts for Streetwear:')
print(reason_counts.head())
print(f'Most frequent (among options): {top_reason}')
print(f'Correct option: {q3_answer}) {q3_options[q3_answer]}')

Q3 -> Return reason counts for Streetwear:
return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
Most frequent (among options): wrong_size
Correct option: B) wrong_size


### **Q4.** Trong `web_traffic.csv`, nguồn truy cập (traffic_source) nào có **tỷ lệ thoát trung bình** (bounce_rate) **thấp nhất** trên tất cả các ngày xuất hiện nguồn đó?

* A) organic_search
* B) paid_search
* C) email_campaign
* D) social_media

In [16]:
source_bounce = web_traffic.groupby('traffic_source', dropna=False)['bounce_rate'].mean()

q4_options = {'A': 'organic_search', 'B': 'paid_search', 'C': 'email_campaign', 'D': 'social_media'}
option_bounce = source_bounce.reindex(list(q4_options.values())).dropna()
best_source = option_bounce.idxmin()
q4_answer = next((k for k, v in q4_options.items() if v == best_source), None)

print('Q4 -> Mean bounce_rate by traffic_source:')
print(source_bounce.sort_values())
print(f'Lowest source (among options): {best_source}')
print(f'Correct option: {q4_answer}) {q4_options[q4_answer]}')

Q4 -> Mean bounce_rate by traffic_source:
traffic_source
email_campaign    0.004458
social_media      0.004476
paid_search       0.004478
referral          0.004499
organic_search    0.004504
direct            0.004511
Name: bounce_rate, dtype: float64
Lowest source (among options): email_campaign
Correct option: C) email_campaign


### **Q5.** Tỷ lệ phần trăm các dòng trong `order_items.csv` có áp dụng khuyến mãi (tức là `promo_id` không null) xấp xỉ là bao nhiêu?

* A) 12%
* B) 25%
* C) 39%
* D) 54%

In [17]:
promo_rate = order_items['promo_id'].notna().mean() * 100

q5_options = {'A': 12, 'B': 25, 'C': 39, 'D': 54}
q5_answer = min(q5_options, key=lambda k: abs(q5_options[k] - promo_rate))

print(f'Q5 -> Promotion usage rate = {promo_rate:.2f}%')
print(f'Closest option: {q5_answer}) {q5_options[q5_answer]}%')

Q5 -> Promotion usage rate = 38.66%
Closest option: C) 39%


### **Q6.** Trong `customers.csv`, xét các khách hàng có `age_group` khác null, nhóm tuổi nào có **số đơn hàng trung bình trên mỗi khách hàng cao nhất**? (tổng số đơn / số khách hàng trong nhóm)

* A) 55+
* B) 25-34
* C) 35-44
* D) 45-54

In [18]:
orders_age = orders[['order_id', 'customer_id']].merge(
    customers[['customer_id', 'age_group']],
    on='customer_id',
    how='left'
)
orders_age = orders_age[orders_age['age_group'].notna()]

orders_by_age = orders_age.groupby('age_group')['order_id'].nunique()
customers_by_age = customers[customers['age_group'].notna()].groupby('age_group')['customer_id'].nunique()
orders_per_customer = (orders_by_age / customers_by_age).sort_values(ascending=False)

q6_options = {'A': '55+', 'B': '25-34', 'C': '35-44', 'D': '45-54'}
option_ratio = orders_per_customer.reindex(list(q6_options.values())).dropna()
best_age_group = option_ratio.idxmax()
q6_answer = next((k for k, v in q6_options.items() if v == best_age_group), None)

print('Q6 -> Avg orders per customer by age_group:')
print(orders_per_customer)
print(f'Best age_group (among options): {best_age_group}')
print(f'Correct option: {q6_answer}) {q6_options[q6_answer]}')

Q6 -> Avg orders per customer by age_group:
age_group
55+      5.406851
45-54    5.357241
35-44    5.337343
25-34    5.245226
18-24    5.226656
dtype: float64
Best age_group (among options): 55+
Correct option: A) 55+


### **Q7.** Vùng (region) nào trong `geography.csv` tạo ra **tổng doanh thu cao nhất** trong `sales_train.csv`?

* A) West
* B) Central
* C) East
* D) Cả ba vùng có doanh thu xấp xỉ bằng nhau

In [19]:
if {'region', 'Revenue'}.issubset(sales_train.columns):
    region_revenue = sales_train.groupby('region', dropna=False)['Revenue'].sum()
else:
    # sales_train.csv/sales.csv does not contain region, so use order-level equivalent revenue.
    order_revenue = orders[['order_id', 'zip']].merge(
        payments[['order_id', 'payment_value']],
        on='order_id',
        how='inner'
    )
    region_revenue = order_revenue.merge(
        geography[['zip', 'region']],
        on='zip',
        how='left'
    ).groupby('region', dropna=False)['payment_value'].sum()

region_revenue = region_revenue.reindex(['West', 'Central', 'East']).dropna()

q7_options = {'A': 'West', 'B': 'Central', 'C': 'East', 'D': 'Cả ba vùng có doanh thu xấp xỉ bằng nhau'}
spread_ratio = (region_revenue.max() - region_revenue.min()) / region_revenue.max()
if spread_ratio <= 0.03:
    q7_answer = 'D'
else:
    top_region = region_revenue.idxmax()
    q7_answer = next((k for k, v in q7_options.items() if v == top_region), None)

print('Q7 -> Total revenue by region:')
print(region_revenue.sort_values(ascending=False))
print(f'Correct option: {q7_answer}) {q7_options[q7_answer]}')

Q7 -> Total revenue by region:
region
East       7.291151e+09
Central    4.719491e+09
West       3.670227e+09
Name: payment_value, dtype: float64
Correct option: C) East


### **Q8.** Trong các đơn hàng có `order_status = 'cancelled'` trong `orders.csv`, **phương thức thanh toán** nào được sử dụng nhiều nhất?

* A) credit_card
* B) cod
* C) paypal
* D) bank_transfer

In [20]:
cancelled_counts = orders.loc[orders['order_status'] == 'cancelled', 'payment_method'].value_counts(dropna=False)

q8_options = {'A': 'credit_card', 'B': 'cod', 'C': 'paypal', 'D': 'bank_transfer'}
option_counts = cancelled_counts.reindex(list(q8_options.values()), fill_value=0)
top_method = option_counts.idxmax()
q8_answer = next((k for k, v in q8_options.items() if v == top_method), None)

print('Q8 -> Payment method counts for cancelled orders:')
print(cancelled_counts)
print(f'Most frequent method (among options): {top_method}')
print(f'Correct option: {q8_answer}) {q8_options[q8_answer]}')

Q8 -> Payment method counts for cancelled orders:
payment_method
credit_card      28452
cod              15468
paypal            7817
apple_pay         5190
bank_transfer     2535
Name: count, dtype: int64
Most frequent method (among options): credit_card
Correct option: A) credit_card


### **Q9.** Trong bốn kích thước sản phẩm (S, M, L, XL), kích thước nào có **tỷ lệ trả hàng cao nhất** (số bản ghi trong `returns` chia cho số dòng trong `order_items`)?

* A) S
* B) M
* C) L
* D) XL

In [ ]:
sizes = ['S', 'M', 'L', 'XL']

items_size = order_items[['product_id']].merge(products[['product_id', 'size']], on='product_id', how='left')
returns_size = returns[['product_id']].merge(products[['product_id', 'size']], on='product_id', how='left')

denominator = items_size[items_size['size'].isin(sizes)].groupby('size').size().reindex(sizes, fill_value=0)
numerator = returns_size[returns_size['size'].isin(sizes)].groupby('size').size().reindex(sizes, fill_value=0)
return_rate_by_size = (numerator / denominator.replace(0, np.nan)).sort_values(ascending=False)

q9_options = {'A': 'S', 'B': 'M', 'C': 'L', 'D': 'XL'}
best_size = return_rate_by_size.idxmax()
q9_answer = next((k for k, v in q9_options.items() if v == best_size), None)

print('Q9 -> Return-rate by size (returns rows / order_items rows):')
print(return_rate_by_size)
print(f'Highest return-rate size: {best_size}')
print(f'Correct option: {q9_answer}) {q9_options[q9_answer]}')

Q9 -> Return-rate by size (returns rows / order_items rows):
size
S     0.056515
L     0.056250
M     0.055660
XL    0.055200
dtype: float64
Highest return-rate size: S
Correct option: A) S


### **Q10.** Trong `payments.csv`, **kế hoạch trả góp** nào có **giá trị thanh toán trung bình trên mỗi đơn hàng cao nhất**?

* A) 1 kỳ (trả một lần)
* B) 3 kỳ
* C) 6 kỳ
* D) 12 kỳ

In [22]:
avg_payment_by_installment = payments.groupby('installments', dropna=False)['payment_value'].mean().sort_values(ascending=False)

q10_options = {'A': 1, 'B': 3, 'C': 6, 'D': 12}
option_avg = avg_payment_by_installment.reindex(list(q10_options.values())).dropna()
best_installment = int(option_avg.idxmax())
q10_answer = next((k for k, v in q10_options.items() if v == best_installment), None)

print('Q10 -> Mean payment_value by installments:')
print(avg_payment_by_installment)
print(f'Highest average installment plan (among options): {best_installment}')
print(f'Correct option: {q10_answer}) {q10_options[q10_answer]} kỳ')

Q10 -> Mean payment_value by installments:
installments
6     24446.654403
3     24399.635486
12    24245.772694
1     24113.274166
2       708.473729
Name: payment_value, dtype: float64
Highest average installment plan (among options): 6
Correct option: C) 6 kỳ
